# PyField Cl₂ walkthrough

This notebook is a guided run of the bundled Cl₂ smoke test. It also doubles as a regression test — `pytest --nbmake` re-executes every cell, so the notebook is guaranteed to work end-to-end.

What we'll do:

1. Load and inspect `tests/cl2.yaml` (the user-facing config).
2. Run a tiny SA optimization against the bundled ReaxFF + 19 ΔE targets.
3. Plot the cost trace.

Requirements: `pip install -e .[dev]` and a working LAMMPS (`pip install lammps[mpi]`).

## 1. Load the YAML config

In [ ]:
import os
# Run from the repo root so the relative paths inside cl2.yaml resolve.
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'examples' else os.getcwd()
os.chdir(ROOT)

from pyfield.config.loader import load_yaml
cfg = load_yaml('tests/cl2.yaml')
print(f'forcefield: {cfg.forcefield.path}')
print(f'structures: {len(cfg.structures)}')
print(f'simulations: {len(cfg.simulations)}')
print(f'targets: {len(cfg.targets)}')
print(f'optimizer: {cfg.optimizer.method}, T={cfg.optimizer.T}, max_iter={cfg.optimizer.max_iter}, seed={cfg.optimizer.seed}')

## 2. Inspect a few structures and targets

The 20 structures are Cl₂ at different bond lengths; the 19 targets are ΔE-from-equilibrium energy combinations.

In [ ]:
for name in list(cfg.structures)[:3]:
    s = cfg.structures[name]
    print(f'  {name}: box={s.box}, atoms={[(a.element, a.z) for a in s.atoms]}')
for t in cfg.targets[:3]:
    extras = t.__pydantic_extra__
    print(f'  target kind={t.kind} weight={t.weight} extras={extras}')

## 3. Run the SA optimization

The same code path as `pyfield run tests/cl2.yaml`. Should finish in well under a second.

In [ ]:
from pyfield.io.lammps import preload_libmpi
preload_libmpi()
from pyfield.optimizers.sa import run_sa
result = run_sa(cfg)
print(f'final cost: {result.final_cost}')
print(f'best ffield: {result.best_ffield_path}')
print(f'cost trace length: {len(result.cost_trace)}')

## 4. Plot the cost trace

In [ ]:
import matplotlib
matplotlib.use('Agg')   # so the notebook test runs in headless CI
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(result.cost_trace, marker='o')
ax.set_xlabel('iteration')
ax.set_ylabel('cost')
ax.set_title('Cl₂ ReaxFF refit — cost trace')
fig.tight_layout()
fig.savefig('examples/_cl2_cost_trace.png', dpi=80)

## 5. Sanity check

If you ran this with the bundled `cl2.yaml` (`seed: 0`), the final cost should be `32907.21505210572`.

In [ ]:
expected = 32907.21505210572
assert abs(result.final_cost - expected) < 1e-6, (result.final_cost, expected)
print('OK — bit-reproducible.')